## 06 — SARIMAX
Huấn luyện SARIMAX với tham số tốt nhất ARIMA(1,1,0)(0,1,1)[3] và dự báo Q2/2026.

In [ ]:
# Tải tham số SARIMAX
ps = pd.read_csv("best_params/best_params_sarimax.csv", index_col=0)
p, d, q = int(ps.loc["p"][0]), int(ps.loc["d"][0]), int(ps.loc["q"][0])
P, D, Q = int(ps.loc["P"][0]), int(ps.loc["D"][0]), int(ps.loc["Q"][0])
m_s = 3

print(f"ARIMA order    : ({p},{d},{q})")
print(f"Seasonal order : ({P},{D},{Q})[{m_s}]")

In [ ]:
# Huấn luyện SARIMAX
model_s = pm.ARIMA(order=(p,d,q), seasonal_order=(P,D,Q,m_s), suppress_warnings=True)
model_s.fit(training["y"].values)
print("SARIMAX fit xong.")

In [ ]:
# Dự báo Q2/2026
sarimax_vals  = model_s.predict(n_periods=3)
pred_sarimax  = pd.DataFrame({"ds": months_q2, "yhat": sarimax_vals})
pred_sarimax["tháng"] = pred_sarimax["ds"].dt.strftime("T%m/%Y")

print("Dự báo SARIMAX Q2/2026:")
for _, r in pred_sarimax.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ VND")

In [ ]:
# Biểu đồ Fitted vs Actual
fitted = model_s.predict_in_sample()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(training.ds, training.y / 1e9, marker="o", linewidth=2, label="Thực tế")
ax.plot(training.ds, fitted / 1e9, linestyle="--", linewidth=2, label="Fitted")
ax.set_title("SARIMAX — Thực tế vs Fitted")
ax.set_xlabel("Tháng")
ax.set_ylabel("Doanh số (tỷ VND)")
ax.legend()
ax.grid(axis="y", alpha=0.4)
fig.tight_layout()
plt.show()